In [3]:
# ============================================================
# IDENTIFY NEXT BATCH — LOCAL EXECUTION
# ============================================================

import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F
import nbformat
from nbconvert import PythonExporter

# Load environment
def run_nb(path):
    with open(path, "r") as f:
        nb = nbformat.read(f, as_version=4)
    src, _ = PythonExporter().from_notebook_node(nb)
    exec(src, globals())

run_nb("/Users/manoharazalki/F1-Analytics/notebooks/00-common/01_environment_config.ipynb")

control_file = f"{CONTROL_PATH}/batch_control/data.parquet"

===== F1 Analytics Environment Configuration =====
BASE_PATH:        /Users/manoharazalki/F1-Analytics/data
LANDING_PATH:     /Users/manoharazalki/F1-Analytics/data/landing
BRONZE_PATH:      /Users/manoharazalki/F1-Analytics/data/bronze
SILVER_PATH:      /Users/manoharazalki/F1-Analytics/data/silver
GOLD_PATH:        /Users/manoharazalki/F1-Analytics/data/gold
INCREMENTAL_PATH: /Users/manoharazalki/F1-Analytics/data/incremental
CONTROL_PATH:     /Users/manoharazalki/F1-Analytics/data/control


In [4]:
# -----------------------------------------
# 1. Receive batch_id from previous notebook
# -----------------------------------------
p_batch_id = next_batch   # passed from notebook 01

if not p_batch_id:
    raise Exception("❌ No batch_id provided")

NameError: name 'next_batch' is not defined

In [ ]:
# -----------------------------------------
# 2. Append new in_progress batch
# -----------------------------------------
df = (
    spark.createDataFrame([Row(batch_id=p_batch_id, status="in_progress")])
        .withColumn("created_timestamp", F.current_timestamp())
        .withColumn("updated_timestamp", F.current_timestamp())
)

df.write.mode("append").parquet(control_file)

print(f"✔ Marked batch {p_batch_id} as in_progress")